<a href="https://colab.research.google.com/github/charitykayy/dsc-course0-m8-lab/blob/main/Aviation_Accidents_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [6]:
import pandas as pd
df = pd.read_csv("/content/AviationData.csv", encoding='latin1', low_memory=False)

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Display the first few rows of the DataFrame
display(df.head())

# Get information about the DataFrame, including data types and non-null counts
print("\nDataFrame Info:")
df.info()

# Get summary statistics for numerical columns
print("\nDataFrame Description:")
display(df.describe())

# Check for missing values
print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

NameError: name 'df' is not defined

In [7]:
# Convert 'Event.Date' to datetime objects
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter data from 1983 onwards as per client's request
df_filtered = df[df['Event.Date'].dt.year >= 1983].copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"Filtered DataFrame shape (1983 onwards): {df_filtered.shape}")
display(df_filtered.head())

Original DataFrame shape: (88889, 31)
Filtered DataFrame shape (1983 onwards): (85289, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
3600,20001214X42040,Accident,LAX83LA093,1983-01-01,"ARROYO GRANDE, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,0.0,1.0,0.0,1.0,VMC,Landing,Probable Cause,NaN
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,Personal,NaN,0.0,0.0,1.0,3.0,VMC,Approach,Probable Cause,NaN
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,Personal,NaN,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,Personal,NaN,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,NaN
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,Personal,NaN,0.0,0.0,2.0,0.0,VMC,Approach,Probable Cause,NaN


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [8]:
# Fill NaN values in injury-related columns with 0, assuming no entry means zero injuries/uninjured
df_filtered['Total.Fatal.Injuries'] = df_filtered['Total.Fatal.Injuries'].fillna(0)
df_filtered['Total.Serious.Injuries'] = df_filtered['Total.Serious.Injuries'].fillna(0)
df_filtered['Total.Minor.Injuries'] = df_filtered['Total.Minor.Injuries'].fillna(0)
df_filtered['Total.Uninjured'] = df_filtered['Total.Uninjured'].fillna(0)

# Derived Column: Total.Occupants
# This column represents the total number of people on board the aircraft during the event.
df_filtered['Total.Occupants'] = (
    df_filtered['Total.Fatal.Injuries'] +
    df_filtered['Total.Serious.Injuries'] +
    df_filtered['Total.Minor.Injuries'] +
    df_filtered['Total.Uninjured']
)

# Derived Column: Total.Fatal.Serious.Injuries
# This column sums up the fatal and serious injuries.
df_filtered['Total.Fatal.Serious.Injuries'] = (
    df_filtered['Total.Fatal.Injuries'] +
    df_filtered['Total.Serious.Injuries']
)

# Derived Column: Fatal.Serious.Injury.Likelihood
# This metric quantifies the proportion of fatal or serious injuries relative to the total occupants.
# A small constant (epsilon) is added to the denominator to prevent division by zero for events with 0 occupants.
epsilon = 1e-6 # A small value to avoid division by zero
df_filtered['Fatal.Serious.Injury.Likelihood'] = (
    df_filtered['Total.Fatal.Serious.Injuries'] /
    (df_filtered['Total.Occupants'] + epsilon)
)

# Display the first few rows with the new columns to verify
display(df_filtered[['Total.Fatal.Injuries', 'Total.Serious.Injuries',
                     'Total.Minor.Injuries', 'Total.Uninjured',
                     'Total.Occupants', 'Total.Fatal.Serious.Injuries',
                     'Fatal.Serious.Injury.Likelihood']].head())

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Occupants,Total.Fatal.Serious.Injuries,Fatal.Serious.Injury.Likelihood
3600,0.0,1.0,0.0,1.0,2.0,1.0,0.5
3601,0.0,0.0,1.0,3.0,4.0,0.0,0.0
3602,0.0,0.0,0.0,2.0,2.0,0.0,0.0
3603,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3604,0.0,0.0,2.0,0.0,2.0,0.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [10]:
# Inspect unique values and their counts in 'Aircraft.Damage'
# It seems there was a KeyError, let's first check the available columns
# print(df_filtered.columns)
# Assuming the correct column name is 'Aircraft.damage' (lowercase 'd') based on typical dataset naming conventions if 'Aircraft.Damage' was not found.
# If still not found, I'd need to inspect the original df or ask for column names.

# Let's assume the column is actually 'Aircraft.damage' for now, or check for similar names.
# For the purpose of this correction, I will assume a common typo or naming convention difference and try 'Aircraft.damage'.
# If that's not it, a deeper inspection of df.columns would be needed.

# For the purpose of this example, I'll demonstrate how to handle if the column was actually 'Aircraft.damage'.
# However, to properly fix it, I need to know the *actual* column name from the original data or an earlier step where columns were printed.
# Since the traceback clearly indicates 'Aircraft.Damage' was tried, and failed, I will need to look for a similar column.
# Let's inspect the columns of df_filtered to confirm the actual name.

# *** Re-evaluating the available columns based on the kernel state ***
# Looking at the `df` and `df_filtered` display, I can see columns like 'Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date'.
# The original problem statement mentions 'Aircraft.Damage', so it should exist. Let's re-examine `df.info()` or `df.columns` to be absolutely sure.
# Since I don't have df.columns output directly, I will assume a case sensitivity issue or a slightly different name.
# A common practice is to standardize column names. I will attempt to access 'Aircraft.Damage' again, but first verify its existence.

# Temporarily, to prevent error and proceed, I'll add a check or print columns.
# However, the instruction is to fix the error assuming the column *should* exist and it's a direct KeyError.
# Without an explicit list of columns from df_filtered, I cannot definitively say what the *correct* column name is.
# Let's assume for a moment the column might be 'Aircraft_Damage' or 'Aircraft Damage'.
# A more robust solution would be to print df_filtered.columns.tolist() first if the column is truly missing.

# Given the KeyError, the most likely scenario is a case mismatch or a slightly different name.
# Let's try to list the columns to identify the correct one.
# Adding a line to print all columns to diagnose, then fix it.

# Original code:
# print(df_filtered['Aircraft.Damage'].value_counts(dropna=False))

# Diagnosis step:
# print(df_filtered.columns.tolist())

# Let's assume from the prompt and typical dataset that 'Aircraft.Damage' is the intended column and there might be a subtle data loading issue or a case issue not immediately apparent from the truncated display.
# If the column truly doesn't exist, the upstream data loading or filtering might have dropped it. This is unlikely for a core column.
# I will attempt to locate the column by searching for 'aircraft' in column names in a case-insensitive way if direct access fails.

# *** Correction based on typical column naming issues ***
# Given the original problem statement and the error, the most common causes are:
# 1. Column does not exist at all (unlikely for a key column as per the problem description).
# 2. Case sensitivity or subtle spelling difference.
# 3. Column was dropped during a previous cleaning step.

# Since the error happened immediately on the first line, it's about accessing it. I cannot fix it without knowing the actual column name.
# As an AI, I should inspect the dataframe's columns to find the exact name.

# Let's try to get a list of columns to diagnose:
print("Available columns in df_filtered:")
print(df_filtered.columns.tolist())

# To provide a fix for the immediate error, I will try to find a column that contains 'damage' (case-insensitive) and use that. If multiple, it will pick the first one.
damage_column = None
for col in df_filtered.columns:
    if 'damage' in col.lower():
        damage_column = col
        break

if damage_column is None:
    print("Error: No column related to 'damage' found in the DataFrame. Please check the dataset.")
else:
    print(f"Found damage column: {damage_column}. Proceeding with this column.")

    # Inspect unique values and their counts in the identified damage column
    print(f"Unique values in '{damage_column}' before cleaning:")
    print(df_filtered[damage_column].value_counts(dropna=False))

    # Standardize values: Ensure consistency (e.g., 'Destroyed' vs. 'Substantial')
    df_filtered[damage_column] = df_filtered[damage_column].replace({
        'Destroyed': 'Destroyed',
        'Substantial': 'Substantial',
        'Minor': 'Minor',
        'None': 'None',
        'Unknown': 'Unknown'
    })

    # Fill NaN values in the damage column with 'Unknown'
    df_filtered[damage_column] = df_filtered[damage_column].fillna('Unknown')

    # Create derived column: 'Aircraft.Destroyed'
    # This column will be True if the damage column value is 'Destroyed', False otherwise.
    df_filtered['Aircraft.Destroyed'] = (df_filtered[damage_column] == 'Destroyed')

    print(f"\nUnique values in '{damage_column}' after cleaning:")
    print(df_filtered[damage_column].value_counts(dropna=False))

    # Display the first few rows with the new column to verify
    display(df_filtered[[damage_column, 'Aircraft.Destroyed']].head())

Available columns in df_filtered:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Occupants', 'Total.Fatal.Serious.Injuries', 'Fatal.Serious.Injury.Likelihood']
Found damage column: Aircraft.damage. Proceeding with this column.
Unique values in 'Aircraft.damage' before cleaning:
Aircraft.damage
Substantial    61775
Destroyed      17575
NaN             3138
Minor           2682
Unknown          119
Name: count, dtype: int64

Unique values in 'Aircraft.damage' after cleaning:
Aircraft.damage
Sub

,Aircraft.damage,Aircraft.Destroyed
3600,Unknown,False
3601,Substantial,False
3602,Substantial,False
3603,Substantial,False
3604,Substantial,False


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [11]:
# --- Cleaning Tasks for 'Make' Column ---

# 1. Inspect unique values and their counts to identify inconsistencies
print("Unique values in 'Make' before cleaning (top 50):")
print(df_filtered['Make'].value_counts(dropna=False).head(50))

# 2. Standardize case: Convert all 'Make' entries to uppercase to handle case variations
df_filtered['Make'] = df_filtered['Make'].str.upper()

# 3. Trim whitespace from 'Make' entries
df_filtered['Make'] = df_filtered['Make'].str.strip()

# 4. Handle common misspellings or variations (example, add more as needed upon inspection)
# This step requires manual review of value_counts() output.
# For demonstration, let's assume 'CESSNA' and 'CEESNA' are the same, or 'BOEING' and 'BOENG'
# You would typically identify these from the value_counts() output.
replacements = {
    'CEESNA': 'CESSNA',
    'BEECH': 'BEECHCRAFT',
    'PIPER AIRCRAFT': 'PIPER',
    'LOCKHEED AIRCRAFT': 'LOCKHEED'
}
df_filtered['Make'] = df_filtered['Make'].replace(replacements)

# 5. Fill NaN values in 'Make' with 'UNKNOWN' or drop them if they are not useful
# For this analysis, we'll fill with 'UNKNOWN' to keep the records.
df_filtered['Make'] = df_filtered['Make'].fillna('UNKNOWN')

# 6. Filter 'Makes' by frequency: Keep only makes with at least 50 occurrences
make_counts = df_filtered['Make'].value_counts()
popular_makes = make_counts[make_counts >= 50].index
df_filtered = df_filtered[df_filtered['Make'].isin(popular_makes)].copy()

print("\nUnique values in 'Make' after cleaning and filtering (top 50):")
print(df_filtered['Make'].value_counts(dropna=False).head(50))

print(f"\nDataFrame shape after cleaning and filtering 'Make': {df_filtered.shape}")

Unique values in 'Make' before cleaning (top 50):
Make
Cessna                            20892
Piper                             11301
CESSNA                             4922
Beech                              4067
PIPER                              2841
Bell                               2022
Boeing                             1545
BOEING                             1151
BEECH                              1042
Mooney                             1036
Grumman                             990
Robinson                            923
Bellanca                            824
Hughes                              742
Schweizer                           612
BELL                                588
Air Tractor                         577
Mcdonnell Douglas                   521
Aeronca                             458
Maule                               428
Champion                            416
De Havilland                        367
Aero Commander                      343
Stinson                  

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [12]:
# 1. Get rid of any NaNs in the 'Model' column
df_filtered['Model'] = df_filtered['Model'].fillna('UNKNOWN')

# 2. Inspect the column and counts for each model
print("Unique values in 'Model' before checking make-model uniqueness (top 50):")
print(df_filtered['Model'].value_counts(dropna=False).head(50))

# 3. Check if model labels are unique to each make
# Group by 'Make' and 'Model' and count occurrences. If count > 1 for a 'Make' and 'Model' combination,
# it means that model appears for multiple makes, but we are looking for the same model name under *different* makes.
# A simpler check is to see if a 'Model' name exists for more than one 'Make'.
model_make_combinations = df_filtered.groupby('Model')['Make'].nunique()
non_unique_models = model_make_combinations[model_make_combinations > 1]

if not non_unique_models.empty:
    print("\nSome 'Model' labels are not unique to each 'Make'. Examples:")
    # Display some examples of models that appear in multiple makes
    for model_name in non_unique_models.head(5).index:
        print(f"  Model '{model_name}' appears in Makes: {df_filtered[df_filtered['Model'] == model_name]['Make'].unique().tolist()}")

    # 4. Create a derived column that is a unique identifier for a given plane type
    df_filtered['Make_Model_Unique'] = df_filtered['Make'] + ' - ' + df_filtered['Model']
    print("\nCreated 'Make_Model_Unique' column to uniquely identify plane types.")
    print("Unique values in 'Make_Model_Unique' (top 10):")
    print(df_filtered['Make_Model_Unique'].value_counts(dropna=False).head(10))
else:
    print("\nAll 'Model' labels appear to be unique to their respective 'Make's. No 'Make_Model_Unique' column needed.")

# Display the first few rows with the new column (if created) to verify
if 'Make_Model_Unique' in df_filtered.columns:
    display(df_filtered[['Make', 'Model', 'Make_Model_Unique']].head())
else:
    display(df_filtered[['Make', 'Model']].head())

Unique values in 'Model' before checking make-model uniqueness (top 50):
Model
152           2231
172           1654
172N          1096
PA-28-140      869
172M           760
150            752
172P           665
182            618
180            596
150M           553
PA-18          550
PA-18-150      550
PA-28-180      548
PA-28-161      526
PA-28-181      509
737            489
206B           488
150L           436
A36            430
PA-38-112      427
G-164A         423
140            385
206            375
170B           374
172S           371
G-164B         366
R44            360
PA-32-300      337
182P           333
PA-24-250      332
269C           328
PA-28R-200     322
PA-12          310
A188B          304
PA-23-250      292
SR22           280
PA28           279
7AC            278
177            276
M20J           275
A185F          271
PA-22-150      270
185            268
R22            266
PA-31-350      265
182Q           257
150F           256
7GCBC          252
7ECA     

,Make,Model,Make_Model_Unique
3601,CESSNA,182P,CESSNA - 182P
3602,CESSNA,182RG,CESSNA - 182RG
3603,CESSNA,182P,CESSNA - 182P
3604,PIPER,PA-28R-200,PIPER - PA-28R-200
3605,CESSNA,140,CESSNA - 140


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

In [13]:
# --- Cleaning Tasks for 'Engine.Type' Column ---
print("\n--- Engine.Type ---")
print("Unique values in 'Engine.Type' before cleaning:")
print(df_filtered['Engine.Type'].value_counts(dropna=False))

df_filtered['Engine.Type'] = df_filtered['Engine.Type'].str.upper().str.strip()
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].replace({
    'RECIPROCATING': 'RECIP',
    'TURBO PROP': 'TURBOPROP',
    'TURBO SHAFT': 'TURBOSHAFT',
    'TURBO FAN': 'TURBOFAN',
    'NONE': 'NO ENGINE',
    'UNKNOWN': 'UNKNOWN'
}).fillna('UNKNOWN') # Fill NaNs with 'UNKNOWN' for consistency
print("Unique values in 'Engine.Type' after cleaning:")
print(df_filtered['Engine.Type'].value_counts(dropna=False))

# --- Cleaning Tasks for 'Weather.Condition' Column ---
print("\n--- Weather.Condition ---")
print("Unique values in 'Weather.Condition' before cleaning:")
print(df_filtered['Weather.Condition'].value_counts(dropna=False))

df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].str.upper().str.strip()
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].replace({
    'VMC': 'VMC',
    'IMC': 'IMC',
    'UNK': 'UNKNOWN',
    'UNKNOWN': 'UNKNOWN'
}).fillna('UNKNOWN') # Fill NaNs with 'UNKNOWN'
print("Unique values in 'Weather.Condition' after cleaning:")
print(df_filtered['Weather.Condition'].value_counts(dropna=False))

# --- Cleaning Tasks for 'Number.of.Engines' Column ---
print("\n--- Number.of.Engines ---")
print("Unique values in 'Number.of.Engines' before cleaning:")
print(df_filtered['Number.of.Engines'].value_counts(dropna=False))

# Convert to numeric, coerce errors to NaN (which will be handled later if needed or left as is)
df_filtered['Number.of.Engines'] = pd.to_numeric(df_filtered['Number.of.Engines'], errors='coerce')
# For this task, we explicitly avoid imputing NaNs unless a specific strategy is given
print("Unique values in 'Number.of.Engines' after cleaning (numeric conversion):")
print(df_filtered['Number.of.Engines'].value_counts(dropna=False))

# --- Cleaning Tasks for 'Purpose.of.flight' Column ---
print("\n--- Purpose.of.flight ---")
print("Unique values in 'Purpose.of.flight' before cleaning:")
print(df_filtered['Purpose.of.flight'].value_counts(dropna=False).head(20))

df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].str.upper().str.strip()
# Consolidate common flight purposes. This list can be expanded upon further inspection.
flight_purpose_replacements = {
    'PERSONAL': 'PERSONAL',
    'INSTRUCTIONAL': 'INSTRUCTIONAL',
    'BUSINESS': 'BUSINESS',
    'UNKNOWN': 'UNKNOWN',
    'FERRY': 'FERRY',
    'OTHER': 'OTHER',
    'POSITIONING': 'FERRY', # Positioning is similar to Ferry
    'PLEASURE': 'PERSONAL', # Pleasure is a type of personal flight
    'NONCOMMERCIAL': 'OTHER'
}
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].replace(flight_purpose_replacements).fillna('UNKNOWN')
# Group less frequent purposes into 'OTHER' or similar, keeping top categories
purpose_counts = df_filtered['Purpose.of.flight'].value_counts()
# Define a threshold for 'Other' or inspect manually
purpose_threshold = 100 # Example threshold
other_purposes = purpose_counts[purpose_counts < purpose_threshold].index
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].replace(other_purposes, 'OTHER')

print("Unique values in 'Purpose.of.flight' after cleaning:")
print(df_filtered['Purpose.of.flight'].value_counts(dropna=False).head(20))

# --- Cleaning Tasks for 'Broad.phase.of.flight' Column ---
print("\n--- Broad.phase.of.flight ---")
print("Unique values in 'Broad.phase.of.flight' before cleaning:")
print(df_filtered['Broad.phase.of.flight'].value_counts(dropna=False))

df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].str.upper().str.strip()
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].replace({
    'TAKEOFF': 'TAKEOFF',
    'LANDING': 'LANDING',
    'CRUISE': 'CRUISE',
    'MANEUVERING': 'MANEUVERING',
    'APPROACH': 'APPROACH',
    'CLIMB': 'CLIMB',
    'DESCENT': 'DESCENT',
    'UNKNOWN': 'UNKNOWN',
    'STANDING': 'GROUND',
    'PARKING': 'GROUND',
    'TAXI': 'GROUND'
}).fillna('UNKNOWN') # Fill NaNs with 'UNKNOWN'
print("Unique values in 'Broad.phase.of.flight' after cleaning:")
print(df_filtered['Broad.phase.of.flight'].value_counts(dropna=False))


--- Engine.Type ---
Unique values in 'Engine.Type' before cleaning:
Engine.Type
Reciprocating      55670
NaN                 5660
Turbo Shaft         2997
Turbo Prop          2807
Turbo Fan           2189
Unknown             1572
Turbo Jet            545
Geared Turbofan       12
LR                     1
UNK                    1
NONE                   1
Name: count, dtype: int64
Unique values in 'Engine.Type' after cleaning:
Engine.Type
RECIP              55670
UNKNOWN             7232
TURBOSHAFT          2997
TURBOPROP           2807
TURBOFAN            2189
TURBO JET            545
GEARED TURBOFAN       12
LR                     1
UNK                    1
NO ENGINE              1
Name: count, dtype: int64

--- Weather.Condition ---
Unique values in 'Weather.Condition' before cleaning:
Weather.Condition
VMC    61322
IMC     5293
NaN     3863
UNK      745
Unk      232
Name: count, dtype: int64
Unique values in 'Weather.Condition' after cleaning:
Weather.Condition
VMC        61322
IMC  

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [14]:
print("Columns before dropping based on NaNs:", df_filtered.shape[1])

# Calculate the percentage of missing values for each column
nan_percentages = df_filtered.isnull().sum() / len(df_filtered) * 100

# Define a threshold for dropping columns (e.g., 50% missing data)
threshold = 50

# Identify columns to drop
columns_to_drop = nan_percentages[nan_percentages > threshold].index.tolist()

print(f"\nColumns with more than {threshold}% missing values:\n{nan_percentages[nan_percentages > threshold].sort_values(ascending=False)}")

# Drop the identified columns from df_filtered
df_filtered = df_filtered.drop(columns=columns_to_drop)

print("\nColumns after dropping based on NaNs:", df_filtered.shape[1])
print("\nUpdated DataFrame info after dropping columns:")
df_filtered.info()

Columns before dropping based on NaNs: 36

Columns with more than 50% missing values:
Schedule             84.600098
Air.carrier          81.241341
FAR.Description      69.512280
Aircraft.Category    69.282765
Longitude            63.048072
Latitude             63.041075
dtype: float64

Columns after dropping based on NaNs: 30

Updated DataFrame info after dropping columns:
<class 'pandas.core.frame.DataFrame'>
Index: 71455 entries, 3601 to 88888
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Event.Id                         71455 non-null  object        
 1   Investigation.Type               71455 non-null  object        
 2   Accident.Number                  71455 non-null  object        
 3   Event.Date                       71455 non-null  datetime64[ns]
 4   Location                         71406 non-null  object        
 5   Country                     

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [15]:
# Save the cleaned DataFrame to a new CSV file
df_filtered.to_csv('aviation_data_cleaned.csv', index=False)
print("Cleaned DataFrame saved to 'aviation_data_cleaned.csv'")

Cleaned DataFrame saved to 'aviation_data_cleaned.csv'
